# 1. Configuration + Dependencies

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

N_COMPANIES = 1000
N_CONTRACTS = 5000
AVG_CONTRACT_MONTHS = 6

INDUSTRIES = ["tech", "finance", "healthcare", "retail", "manufacturing"]
REGIONS = ["NA", "EU", "APAC"]
PLAN_TIERS = ["Basic", "Professional", "Enterprise"]

PRICE_MAP = {
    "Basic": 25,
    "Professional": 60,
    "Enterprise": 120
}

# 2. Latent Variables

In [2]:
latent = pd.DataFrame({
    "company_id": np.arange(N_COMPANIES),
    "product_fit": np.random.normal(0, 1, N_COMPANIES),
    "customer_success_quality": np.random.normal(0, 1, N_COMPANIES),
    "sales_process_maturity": np.random.normal(0, 1, N_COMPANIES),
    "budget_pressure": np.random.normal(0, 1, N_COMPANIES)
})

# 3. Companies

In [3]:
companies = pd.DataFrame({
    "company_id": np.arange(N_COMPANIES),
    "company_size": np.random.lognormal(mean=5, sigma=1, size=N_COMPANIES).astype(int),
    "industry": np.random.choice(INDUSTRIES, N_COMPANIES),
    "region": np.random.choice(REGIONS, N_COMPANIES),
    "growth_rate": np.random.normal(0.1, 0.05, N_COMPANIES)
})

companies["annual_revenue"] = (
    companies["company_size"] *
    np.random.normal(120000, 30000, N_COMPANIES)
)

companies = companies.merge(
    latent[["company_id", "sales_process_maturity"]],
    on="company_id"
)

companies["crm_maturity_score"] = (
    1 / (1 + np.exp(-(0.5 * companies["sales_process_maturity"]
                      + np.random.normal(0, 0.5, N_COMPANIES))))
)

companies.info()
companies.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   company_id              1000 non-null   int64  
 1   company_size            1000 non-null   int64  
 2   industry                1000 non-null   object 
 3   region                  1000 non-null   object 
 4   growth_rate             1000 non-null   float64
 5   annual_revenue          1000 non-null   float64
 6   sales_process_maturity  1000 non-null   float64
 7   crm_maturity_score      1000 non-null   float64
dtypes: float64(4), int64(2), object(2)
memory usage: 62.6+ KB


,company_id,company_size,industry,region,growth_rate,annual_revenue,sales_process_maturity,crm_maturity_score
0,0,62,tech,EU,0.050758,6.463845e+06,-0.675178,0.270580
1,1,143,retail,APAC,0.128424,1.245357e+07,-0.144519,0.510263
2,2,151,manufacturing,APAC,0.141777,1.680633e+07,-0.792420,0.432901
3,3,238,retail,NA,0.067953,8.462937e+06,-0.307962,0.340177
4,4,37,healthcare,NA,0.198458,5.445396e+06,-1.893615,0.231505


# 4. Contracts

In [4]:
contracts = pd.DataFrame({
    "contract_id": np.arange(N_CONTRACTS),
    "company_id": np.random.choice(companies["company_id"], N_CONTRACTS)
})

contracts = contracts.merge(companies, on="company_id")

contracts["seats_start"] = (
    np.random.poisson(0.12 * contracts["company_size"]) + 5
)

contracts["plan_tier"] = np.random.choice(
    PLAN_TIERS,
    N_CONTRACTS,
    p=[0.5, 0.35, 0.15]
)

contracts["price_per_seat"] = contracts["plan_tier"].map(PRICE_MAP)

contracts["discount_pct"] = np.random.beta(2, 8, N_CONTRACTS)

contracts["contract_value"] = (
    contracts["seats_start"]
    * contracts["price_per_seat"]
    * (1 - contracts["discount_pct"])
)

contracts["duration_months"] = (
    np.random.poisson(AVG_CONTRACT_MONTHS, N_CONTRACTS) + 1
)

contracts.info()
contracts.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   contract_id             5000 non-null   int64  
 1   company_id              5000 non-null   int64  
 2   company_size            5000 non-null   int64  
 3   industry                5000 non-null   object 
 4   region                  5000 non-null   object 
 5   growth_rate             5000 non-null   float64
 6   annual_revenue          5000 non-null   float64
 7   sales_process_maturity  5000 non-null   float64
 8   crm_maturity_score      5000 non-null   float64
 9   seats_start             5000 non-null   int32  
 10  plan_tier               5000 non-null   object 
 11  price_per_seat          5000 non-null   int64  
 12  discount_pct            5000 non-null   float64
 13  contract_value          5000 non-null   float64
 14  duration_months         5000 non-null   

,contract_id,company_id,company_size,industry,region,growth_rate,annual_revenue,sales_process_maturity,crm_maturity_score,seats_start,plan_tier,price_per_seat,discount_pct,contract_value,duration_months
0,0,598,226,tech,EU,0.059017,1.782664e+07,1.200079,0.627441,23,Enterprise,120,0.264037,2031.257870,10
1,1,39,74,retail,NA,0.123574,2.911474e+06,-1.769076,0.278178,15,Professional,60,0.089815,819.166421,12
2,2,160,150,manufacturing,NA,0.055199,1.409863e+07,-1.111458,0.611711,15,Professional,60,0.279983,648.015483,13
3,3,702,288,retail,NA,0.131426,3.436947e+07,2.012270,0.695971,46,Basic,25,0.033091,1111.945865,7
4,4,536,184,retail,NA,0.083454,2.407745e+07,0.975817,0.546493,18,Professional,60,0.089121,983.748806,7


# 5. Monthly Usage

In [5]:
usage_rows = []

for _, row in contracts.iterrows():
    for m in range(row["duration_months"]):
        usage_rows.append({
            "contract_id": row["contract_id"],
            "company_id": row["company_id"],
            "month_index": m,
            "seats_start": row["seats_start"]
        })

usage = pd.DataFrame(usage_rows)

usage = usage.merge(latent, on="company_id")

usage["active_users"] = (
    usage["seats_start"]
    * (1 / (1 + np.exp(-(1.2 * usage["product_fit"]))))
).astype(int)

usage["logins_total"] = (
    usage["active_users"]
    * np.random.poisson(8, len(usage))
)

usage["leads_created"] = (
    usage["active_users"]
    * np.random.poisson(2, len(usage))
)

usage["opportunities_created"] = (
    usage["leads_created"]
    * np.random.uniform(0.3, 0.7, len(usage))
).astype(int)

usage["reports_generated"] = (
    usage["active_users"]
    * np.random.poisson(1, len(usage))
)

usage["feature_adoption_score"] = (
    1 / (1 + np.exp(-(usage["product_fit"])))
)

usage.info()
usage.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34909 entries, 0 to 34908
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   contract_id               34909 non-null  int64  
 1   company_id                34909 non-null  int64  
 2   month_index               34909 non-null  int64  
 3   seats_start               34909 non-null  int64  
 4   product_fit               34909 non-null  float64
 5   customer_success_quality  34909 non-null  float64
 6   sales_process_maturity    34909 non-null  float64
 7   budget_pressure           34909 non-null  float64
 8   active_users              34909 non-null  int64  
 9   logins_total              34909 non-null  int64  
 10  leads_created             34909 non-null  int64  
 11  opportunities_created     34909 non-null  int64  
 12  reports_generated         34909 non-null  int64  
 13  feature_adoption_score    34909 non-null  float64
dtypes: flo

,contract_id,company_id,month_index,seats_start,product_fit,customer_success_quality,sales_process_maturity,budget_pressure,active_users,logins_total,leads_created,opportunities_created,reports_generated,feature_adoption_score
0,0,598,0,23,-0.444293,0.020794,1.200079,0.336145,8,72,32,15,16,0.390718
1,0,598,1,23,-0.444293,0.020794,1.200079,0.336145,8,40,0,0,16,0.390718
2,0,598,2,23,-0.444293,0.020794,1.200079,0.336145,8,72,8,2,16,0.390718
3,0,598,3,23,-0.444293,0.020794,1.200079,0.336145,8,40,16,10,24,0.390718
4,0,598,4,23,-0.444293,0.020794,1.200079,0.336145,8,72,8,2,8,0.390718


# 6. Tickets

In [6]:
tickets = usage[[
    "contract_id",
    "company_id",
    "month_index",
    "logins_total",
    "customer_success_quality"
]].copy()

tickets["ticket_count"] = np.random.poisson(
    0.03 * tickets["logins_total"]
)

tickets["avg_resolution_hours"] = (
    np.random.normal(12, 2, len(tickets))
    - 4 * tickets["customer_success_quality"]
)

tickets["csat_score"] = (
    3.5
    + 0.6 * tickets["customer_success_quality"]
    - 0.02 * tickets["ticket_count"]
    + np.random.normal(0, 0.3, len(tickets))
)

tickets["escalation_count"] = (
    tickets["ticket_count"]
    * np.random.uniform(0.05, 0.15, len(tickets))
).astype(int)

# Introduce missing CSAT values
mask = np.random.rand(len(tickets)) < 0.2
tickets.loc[mask, "csat_score"] = np.nan

tickets.info()
tickets.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34909 entries, 0 to 34908
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   contract_id               34909 non-null  int64  
 1   company_id                34909 non-null  int64  
 2   month_index               34909 non-null  int64  
 3   logins_total              34909 non-null  int64  
 4   customer_success_quality  34909 non-null  float64
 5   ticket_count              34909 non-null  int32  
 6   avg_resolution_hours      34909 non-null  float64
 7   csat_score                27920 non-null  float64
 8   escalation_count          34909 non-null  int64  
dtypes: float64(3), int32(1), int64(5)
memory usage: 2.3 MB


,contract_id,company_id,month_index,logins_total,customer_success_quality,ticket_count,avg_resolution_hours,csat_score,escalation_count
0,0,598,0,72,0.020794,3,10.049976,2.954071,0
1,0,598,1,40,0.020794,1,14.235582,NaN,0
2,0,598,2,72,0.020794,2,9.084398,NaN,0
3,0,598,3,40,0.020794,0,13.221284,NaN,0
4,0,598,4,72,0.020794,1,14.947153,3.594038,0


# 7. Renewal Outcome

In [7]:
usage_summary = usage.groupby("contract_id").agg({
    "logins_total": "mean",
    "active_users": "mean"
}).reset_index()

renewals = contracts[["contract_id", "seats_start"]].merge(
    usage_summary,
    on="contract_id"
)

logit = (
    0.8 * renewals["logins_total"] / 100
    + 0.6 * renewals["active_users"] / 10
    - 0.5 * np.random.rand(len(renewals))
)

renewal_prob = 1 / (1 + np.exp(-logit))

renewals["renewed"] = np.random.binomial(1, renewal_prob)

renewals["seat_change"] = (
    renewals["seats_start"]
    * np.random.uniform(-0.2, 0.5, len(renewals))
).astype(int)

renewals.info()
renewals.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   contract_id   5000 non-null   int64  
 1   seats_start   5000 non-null   int32  
 2   logins_total  5000 non-null   float64
 3   active_users  5000 non-null   float64
 4   renewed       5000 non-null   int32  
 5   seat_change   5000 non-null   int64  
dtypes: float64(2), int32(2), int64(2)
memory usage: 195.4 KB


,contract_id,seats_start,logins_total,active_users,renewed,seat_change
0,0,23,59.200000,8.0,1,2
1,1,15,69.333333,8.0,1,2
2,2,15,25.615385,3.0,1,6
3,3,46,109.571429,13.0,0,5
4,4,18,40.571429,4.0,1,0


# 8. Remove Latent Variables

In [8]:
companies = companies.drop(columns=["sales_process_maturity"])

usage = usage.drop(columns=[
    "product_fit",
    "customer_success_quality",
    "sales_process_maturity",
    "budget_pressure"
])

tickets = tickets.drop(columns=["customer_success_quality"])

# 9. Final formatting + Export

In [9]:
usage_monthly = usage[[
    "contract_id",
    "month_index",
    "logins_total",
    "active_users",
    "leads_created",
    "opportunities_created",
    "reports_generated",
    "feature_adoption_score"
]]

support_tickets_monthly = tickets[[
    "contract_id",
    "month_index",
    "ticket_count",
    "avg_resolution_hours",
    "csat_score",
    "escalation_count"
]]

companies.to_csv("dataset/companies.csv", index=False)
contracts.to_csv("dataset/contracts.csv", index=False)
usage_monthly.to_csv("dataset/usage_monthly.csv", index=False)
support_tickets_monthly.to_csv("dataset/support_tickets_monthly.csv", index=False)
renewals.to_csv("dataset/renewals.csv", index=False)

print("Synthetic dataset generation complete.")
print("Files exported:")
print(" - companies.csv")
print(" - contracts.csv")
print(" - usage_monthly.csv")
print(" - support_tickets_monthly.csv")
print(" - renewals.csv")

Synthetic dataset generation complete.
Files exported:
 - companies.csv
 - contracts.csv
 - usage_monthly.csv
 - support_tickets_monthly.csv
 - renewals.csv
